# Colab connection check
Select kernel → Colab → New Colab Server → GPU (T4), then **Run All**.

In [ ]:
!nvidia-smi

In [ ]:
import os, platform, sys
import torch

print("host:", platform.node(), "| python:", sys.version.split()[0])
print("running on Colab:", os.path.exists("/content"))
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"

p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | {p.total_memory / 1e9:.1f} GB | compute capability {p.major}.{p.minor}")
print("native bf16 (needs capability >= 8.0):", p.major >= 8)

In [ ]:
# Measure real matmul throughput in fp32 and fp16 (the precision we'll train with on a T4).
import time

def tflops(dtype, n=4096, iters=50):
    a = torch.randn(n, n, device="cuda", dtype=dtype)
    b = torch.randn(n, n, device="cuda", dtype=dtype)
    for _ in range(5):
        a @ b
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        a @ b
    torch.cuda.synchronize()
    return 2 * n**3 * iters / (time.perf_counter() - start) / 1e12

print(f"fp32: {tflops(torch.float32):5.1f} TFLOPS")
print(f"fp16: {tflops(torch.float16):5.1f} TFLOPS")
print("matmul ok:", (torch.ones(8, 8, device='cuda') @ torch.ones(8, 8, device='cuda')).sum().item() == 512)